# DIT5411 — Train Model B only (Google Colab)

**Runtime → Change runtime type → T4 GPU.** This notebook trains **Model B** only and lets you download the trained `.keras` file.

Colab's free GPU expires after ~2h, so training is split into **short epochs** (10,000 steps each, ~1.2h). Run the training cell **twice**:
- **Session 1** → trains epoch 1, saves a checkpoint
- **Session 2** → re-run the same cell; `--resume` continues with epoch 2 and saves `model_B.keras`

If a session dies mid-epoch, only that partial epoch is lost — completed checkpoints survive in the session. To survive a full runtime reset, mount Drive in the setup cell and set `OUT_DIR` to a Drive path.

In [ ]:
import os, sys, subprocess, glob, json

# Clone/pull the project code
if not os.path.isfile('train.py'):
    if not os.path.isdir('DIT5411-LuiYuHin'):
        subprocess.run(['git', 'clone', '-q', 'https://github.com/cveg1234/DIT5411-LuiYuHin.git'], check=True)
    os.chdir('DIT5411-LuiYuHin')
subprocess.run(['git', 'pull', '-q'])
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])

# Optional: uncomment to persist checkpoints/results on Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# OUT_DIR = '/content/drive/MyDrive/DIT5411/results'
OUT_DIR = 'results'
os.makedirs(OUT_DIR, exist_ok=True)

import tensorflow as tf
print('Working dir:', os.getcwd())
print('GPU:', tf.config.list_physical_devices('GPU'))
print('Output dir:', OUT_DIR)

## Dataset (one time per runtime)
Downloads and extracts 684,677 images. Skips if already present.

In [ ]:
DATA_DIR = os.path.join(os.getcwd(), 'data', 'cleaned_data')
if not os.path.isdir(DATA_DIR) or len(os.listdir(DATA_DIR)) < 1000:
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/chenkenanalytic/handwritting_data_all.git'], check=True)
    with open('all_data.zip', 'wb') as out:
        for part in sorted(glob.glob('handwritting_data_all/all_data.zip.*')):
            with open(part, 'rb') as f:
                out.write(f.read())
    subprocess.run([sys.executable, 'extract_dataset.py',
                    '--zip', 'all_data.zip', '--dest', DATA_DIR], check=True)
print('Character folders:', len(os.listdir(DATA_DIR)))

## Train Model B (10,000 steps per session)
Run this cell, then **run it again after the first finishes** (or in a new session with `--resume`) to complete epoch 2.

In [ ]:
r = subprocess.run([sys.executable, 'train.py', '--data-dir', DATA_DIR,
                    '--output-dir', OUT_DIR, '--epochs', '2',
                    '--steps-per-epoch', '10000',
                    '--models', 'B', '--resume'])
print('Model B exit code:', r.returncode)

sp = os.path.join(OUT_DIR, 'checkpoints', 'model_B_state.json')
if os.path.isfile(sp):
    st = json.load(open(sp))
    print('Completed epochs:', st['completed_epochs'])
print('model_B.keras saved:', os.path.isfile(os.path.join(OUT_DIR, 'model_B.keras')))

## Results

In [ ]:
from IPython.display import Markdown, display
rp = os.path.join(OUT_DIR, 'RESULTS.md')
display(Markdown(open(rp, encoding='utf-8').read()) if os.path.isfile(rp)
        else 'RESULTS.md not generated yet — finish both epochs first.')

## Download the trained model
Downloads `model_B.keras` to your computer through the browser, or copies it to Google Drive (uncomment the Drive line).

In [ ]:
from google.colab import files
candidates = [os.path.join(OUT_DIR, 'model_B.keras'),
              os.path.join(OUT_DIR, 'checkpoints', 'model_B_latest.keras')]
model_path = next((p for p in candidates if os.path.isfile(p)), None)
if model_path is None:
    print('No model file yet — finish training first.')
else:
    print('Downloading', model_path, '(', round(os.path.getsize(model_path)/1024/1024), 'MB )')
    # To save on Drive instead: shutil.copy2(model_path, '/content/drive/MyDrive/DIT5411/model_B.keras')
    files.download(model_path)